# LogisticRegression — Feature Selection

**Method:** L1 Regularization Path (Lasso) with cross-validated stability analysis.

**Why L1 instead of forward selection?**
- Forward selection is O(N²) — with 60 features, that's 1,830 model fits on a single split
- L1 regularization naturally performs feature selection by driving coefficients to exactly zero
- It handles correlated features gracefully — from a group of correlated features, L1 picks one and zeros the others
- Combined with cross-validation, it gives reliable feature importance that doesn't overfit to one split

**Pipeline:**
1. **Decorrelation**: collapse redundant features (Spearman clustering, |ρ| > 0.7)
2. **L1 path**: scan regularization strength C with GroupKFold(5) cross-validation
3. **Stability analysis**: only keep features selected in ≥60% of folds

**Evaluation:** GroupKFold(n_splits=5) grouped by candle_id — no data leakage between candles.


In [ ]:
import sys

sys.path.insert(0, str(__import__("pathlib").Path.cwd().parent))

import json as _json
import random
import warnings
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from feature_utils import (
    decorrelate_features,
    feature_stability_report,
    plot_correlation_matrix,
    plot_stability,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")
random.seed(42)
np.random.seed(42)

FEATURES_PATH = Path("../../data/latest_features.jsonl")

## 1. Load data

In [ ]:
rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        rows.append(_json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

NON_FEAT = {
    "candle_id",
    "session",
    "timestamp",
    "elapsed_pct",
    "btc_price",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
    "market_volume",
    "outcome",
    "target",
}
all_feat_cols = sorted([c for c in df.columns if c not in NON_FEAT])
df[all_feat_cols] = df[all_feat_cols].fillna(0.0)

print(f"Features: {len(all_feat_cols)}")
print(f"Candles: {df['candle_id'].nunique()}")
print(f"Rows: {len(df):,}")
print(f"UP rate: {df['target'].mean() * 100:.1f}%")

## 2. Stage 1: Decorrelation

Many features are highly correlated — especially the price-proxy features (`up_risk_reward`,
`down_risk_reward`, `rr_spread`, `up_implied_probability`, etc.) which all encode
"which side is the market favorite."

Correlated features cause problems:
- **Inflated importance**: models split credit between redundant features
- **Unstable selection**: which correlated feature gets picked varies by random split
- **Crowding out**: redundant price features fill the top-N, preventing technical indicators from being selected

We fix this by clustering features with Spearman |ρ| > 0.7 and keeping only the cluster
representative with highest mutual information with the target.


In [ ]:
# Show correlation before decorrelation
plot_correlation_matrix(df, all_feat_cols, "Before Decorrelation (all features)")

# Decorrelate
decorr_features, cluster_info = decorrelate_features(
    df,
    all_feat_cols,
    df["target"].values,
    threshold=0.7,
)

print(f"\nBefore: {len(all_feat_cols)} features")
print(f"After:  {len(decorr_features)} features ({len(all_feat_cols) - len(decorr_features)} dropped)")
print("\nClusters with >1 member (collapsed):")
for feat, info in sorted(cluster_info.items(), key=lambda x: -x[1]["cluster_size"]):
    if info["cluster_size"] > 1:
        print(f"  {feat} (MI={info['mutual_info']:.4f}) <- kept from {info['cluster_size']} features")
        print(f"    dropped: {info['dropped']}")

# Show correlation after
plot_correlation_matrix(df, decorr_features, f"After Decorrelation ({len(decorr_features)} features)")

## 3. Stage 2: L1 Regularization Path

L1 (Lasso) regularization adds a penalty proportional to the absolute value of coefficients.
As we increase regularization strength (lower C), more coefficients are driven to exactly zero.

We scan a range of C values using **GroupKFold(5)** cross-validation:
- At each C, train LR with L1 penalty on 4 folds, evaluate on the 5th
- Record which features have non-zero coefficients and the CV accuracy
- The optimal C is the one with highest mean CV accuracy

This replaces the old O(N²) forward selection with an O(N_C × K) approach.


In [ ]:
C_values = np.logspace(-3, 1, 25)
gkf = GroupKFold(n_splits=5)
groups = df["candle_id"].values

results = []
fold_selected_features = {}

for c_val in C_values:
    fold_accs = []
    fold_f1s = []
    fold_briers = []
    fold_feats = []

    for train_idx, val_idx in gkf.split(df, df["target"], groups=groups):
        X_train = df.iloc[train_idx][decorr_features].values
        X_val = df.iloc[val_idx][decorr_features].values
        y_train = df.iloc[train_idx]["target"].values
        y_val = df.iloc[val_idx]["target"].values

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)

        model = LogisticRegression(C=c_val, penalty="l1", solver="saga", max_iter=2000, random_state=42)
        model.fit(X_train, y_train)

        probs = model.predict_proba(X_val)[:, 1]
        preds = (probs >= 0.5).astype(int)

        fold_accs.append(accuracy_score(y_val, preds))
        fold_f1s.append(f1_score(y_val, preds))
        fold_briers.append(brier_score_loss(y_val, probs))

        nonzero = [decorr_features[i] for i in range(len(decorr_features)) if abs(model.coef_[0][i]) > 1e-6]
        fold_feats.append(nonzero)

    results.append(
        {
            "C": c_val,
            "acc_mean": np.mean(fold_accs),
            "acc_std": np.std(fold_accs),
            "f1_mean": np.mean(fold_f1s),
            "brier_mean": np.mean(fold_briers),
            "n_features_mean": np.mean([len(ff) for ff in fold_feats]),
        }
    )
    fold_selected_features[c_val] = fold_feats

results_df = pd.DataFrame(results)

best_idx = results_df["acc_mean"].idxmax()
best_C = results_df.loc[best_idx, "C"]
best_acc = results_df.loc[best_idx, "acc_mean"]
best_std = results_df.loc[best_idx, "acc_std"]
best_n = results_df.loc[best_idx, "n_features_mean"]

print(f"Best C={best_C:.4f}: accuracy={best_acc * 100:.1f}% +/- {best_std * 100:.1f}%, ~{best_n:.0f} features")

### L1 path visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogx(results_df["C"], results_df["acc_mean"] * 100, "o-", color="steelblue")
axes[0].fill_between(
    results_df["C"],
    (results_df["acc_mean"] - results_df["acc_std"]) * 100,
    (results_df["acc_mean"] + results_df["acc_std"]) * 100,
    alpha=0.2,
)
axes[0].axvline(best_C, color="green", linestyle="--", alpha=0.5, label=f"best C={best_C:.4f}")
axes[0].set_xlabel("C (regularization strength)")
axes[0].set_ylabel("CV Accuracy (%)")
axes[0].set_title("L1 Path: Accuracy vs Regularization")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].semilogx(results_df["C"], results_df["n_features_mean"], "o-", color="darkorange")
axes[1].axvline(best_C, color="green", linestyle="--", alpha=0.5)
axes[1].set_xlabel("C (regularization strength)")
axes[1].set_ylabel("Features (mean across folds)")
axes[1].set_title("L1 Path: Feature Count vs Regularization")
axes[1].grid(alpha=0.3)

plt.suptitle("LogisticRegression L1 Feature Selection", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Stage 3: Stability Analysis

A feature selected in only 1 out of 5 folds is noise — it happened to help on that particular
split but doesn't generalize. A feature selected in 5/5 folds is genuine signal.

We take the features selected at the optimal C across all 5 folds, and keep only those
selected in ≥60% of folds (at least 3 out of 5).


In [ ]:
best_fold_feats = fold_selected_features[best_C]

stable_features, stability_scores = feature_stability_report(
    best_fold_feats,
    decorr_features,
    threshold=0.6,
)

plot_stability(stability_scores, threshold=0.6, title="LR Feature Stability (L1 at optimal C)")

print(f"\nStable features (selected in >=60% of folds): {len(stable_features)}")
for f in stable_features:
    print(f"  {f}: {stability_scores[f] * 100:.0f}%")

## 5. Final cross-validated evaluation

In [ ]:
accs, f1s, briers = [], [], []

for train_idx, val_idx in gkf.split(df, df["target"], groups=groups):
    X_train = df.iloc[train_idx][stable_features].values
    X_val = df.iloc[val_idx][stable_features].values
    y_train = df.iloc[train_idx]["target"].values
    y_val = df.iloc[val_idx]["target"].values

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    model = LogisticRegression(C=best_C, penalty="l1", solver="saga", max_iter=2000, random_state=42)
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_val)[:, 1]
    preds = (probs >= 0.5).astype(int)

    accs.append(accuracy_score(y_val, preds))
    f1s.append(f1_score(y_val, preds))
    briers.append(brier_score_loss(y_val, probs))

print(f"Final CV results with {len(stable_features)} stable features:")
print(f"  Accuracy: {np.mean(accs) * 100:.1f}% +/- {np.std(accs) * 100:.1f}%")
print(f"  F1:       {np.mean(f1s) * 100:.1f}% +/- {np.std(f1s) * 100:.1f}%")
print(f"  Brier:    {np.mean(briers):.4f} +/- {np.std(briers):.4f}")

## 6. Save config

In [ ]:
config = {
    "model": "logistic_regression",
    "features": stable_features,
    "n_features": len(stable_features),
    "accuracy_cv_mean": round(float(np.mean(accs)), 4),
    "accuracy_cv_std": round(float(np.std(accs)), 4),
    "f1_cv_mean": round(float(np.mean(f1s)), 4),
    "brier_cv_mean": round(float(np.mean(briers)), 4),
    "selection_method": "l1_regularization_path",
    "cv_folds": 5,
    "stability_threshold": 0.6,
    "feature_stability": {f: round(stability_scores[f], 2) for f in stable_features},
    "decorrelation_threshold": 0.7,
    "features_before_decorrelation": len(all_feat_cols),
    "features_after_decorrelation": len(decorr_features),
    "hyperparameters": {"C": round(float(best_C), 6), "penalty": "l1", "solver": "saga", "max_iter": 2000},
    "source": "data/latest_features.jsonl",
    "created_at": datetime.now(UTC).isoformat(),
}

out_path = Path("../../data/optimal_features_lr.json")
with open(out_path, "w") as f:
    _json.dump(config, f, indent=2)

print(f"Saved {config['n_features']} LR features to {out_path}")
print(f"Features: {config['features']}")

## Conclusion

Feature selection pipeline:
1. **Decorrelation**: collapsed correlated features using Spearman clustering (|ρ| > 0.7)
2. **L1 path**: scanned 25 regularization strengths with GroupKFold(5) cross-validation
3. **Stability**: kept features selected in ≥60% of folds

Run `lr/02_export.ipynb` to export the model, then `lr/03_strategy.ipynb` for strategy discovery.
